In [1]:
import logging
from pyspark import SparkConf
from pyspark import SparkContext
from pyspark.sql import SparkSession
from pyspark.sql.functions import month, days
from pyspark.sql.types import StructType, StructField, LongType, DoubleType, StringType, TimestampNTZType
from datetime import datetime, timedelta

In [2]:
def create_spark_session(app_name: str) -> SparkSession:
    spark = (
        SparkSession.builder
        .master("spark://spark-master:7077")
        .appName(app_name)
        .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
        .config("spark.sql.catalog.lakehouse_prod","org.apache.iceberg.spark.SparkCatalog")
        .config("spark.sql.catalog.lakehouse_prod.type", "hive")
        .config("spark.sql.catalog.lakehouse_prod.uri","thrift://hive-metastore:9083")
        .config("spark.sql.catalog.lakehouse_prod.warehouse","s3a://lakehouse-prod-bucket/warehouse/")
        .config("spark.sql.catalog.lakehouse_prod.io-impl","org.apache.iceberg.aws.s3.S3FileIO")
        .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
        .config("spark.sql.adaptive.enabled", "true")
        .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
        .enableHiveSupport()
        .getOrCreate()
    )

    spark.sparkContext.setLogLevel("ERROR")
    print("NOTE: SparkSession created successfully!")

    return spark

In [3]:
app_name = 'Spark-Iceberg-Test'
spark = create_spark_session(app_name)
spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/14 17:48:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


NOTE: SparkSession created successfully!


In [4]:
file_name = f"sp500_profiles_{datetime.now().strftime('%Y%m%d')}.parquet"
s3_path = f"s3a://datalake-landing/profiles/{file_name}"

catalog_name = "lakehouse_prod"
schema_name = "bronze_db"
table_name = "bronze_sp500_profiles"
bronze_directory = f"s3a://lakehouse-bronze-bucket/warehouse/{schema_name}/{table_name}/"

In [5]:
df = spark.read.parquet(s3_path, header=True, inferSchema=True)
df.show(5)

+--------------------+-------------+-----+----------+-------------+--------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------------+--------------------+---------+---------+----------------+---------------------+-----------+-------------------+-------------------------+--------------------+-------------+------+---------+-------------+------+------+-------+--------------------------+-----------------+-------------------+--------------------+------------+-------------+--------------+-----------+------------------------+-----+----------+---------+---------+-------------------+-------------+-------------------+-----------------------+------+------+-------+-------+-------------+-------------------+---------------+----------------+-----------+----------+----------------------------+---------------+--------------------+--------------------------+-------------

In [6]:
spark.read.parquet(s3_path).printSchema()

root
 |-- address1: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip: string (nullable = true)
 |-- country: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- website: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- industryKey: string (nullable = true)
 |-- industryDisp: string (nullable = true)
 |-- sector: string (nullable = true)
 |-- sectorKey: string (nullable = true)
 |-- sectorDisp: string (nullable = true)
 |-- longBusinessSummary: string (nullable = true)
 |-- fullTimeEmployees: double (nullable = true)
 |-- companyOfficers: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- age: double (nullable = true)
 |    |    |-- exercisedValue: long (nullable = true)
 |    |    |-- fiscalYear: double (nullable = true)
 |    |    |-- maxAge: long (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- title: string (nullable = true)


In [ ]:
# df.writeTo(f"{catalog_name}.{schema_name}.{table_name}") \
#     .tableProperty("location", bronze_directory) \
#     .createOrReplace()

In [7]:
catalog_name = "lakehouse_prod"
schema_name = "bronze_db"
table_name = "bronze_sp500_profiles"

In [10]:
spark.sql(f"""
    SELECT * FROM {catalog_name}.{schema_name}.{table_name} LIMIT 5;
  """).show()

+--------------------+-------------+-----+----------+-------------+--------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------------+--------------------+---------+---------+----------------+---------------------+-----------+-------------------+-------------------------+--------------------+-------------+------+---------+-------------+------+------+-------+--------------------------+-----------------+-------------------+--------------------+------------+-------------+--------------+-----------+------------------------+-----+----------+---------+---------+-------------------+-------------+-------------------+-----------------------+------+------+-------+-------+-------------+-------------------+---------------+----------------+-----------+----------+----------------------------+---------------+--------------------+--------------------------+-------------

In [11]:
# Stop the SparkSession
spark.stop()